# NB4i (v6): why was attack F's evasion identical in all five seeds?

In NB4h, attack F (all 93 features moved independently by 0.1 sigma) gave 0.589 overall and [0.958, 0.990, 0.996, 0, 0] per class in **every** seed, while the cyber-only attacks varied by seed. This notebook finds out whether that is a data ceiling, a shared feature, or a bug.

Per seed (three seeds) it: (1) runs F and records which columns changed in each evaded row; (2) compares the evaded sets across seeds; (3) runs F with the most-used column removed, and with only the 60 cyber features, to see where the evasion comes from; (4) lists the rows nobody can evade. **Needs** the same four files as NB4h. About 15 minutes.

In [ ]:
import os, sys, time, json, hashlib, warnings
import numpy as np, pandas as pd, joblib
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

def find_project_dir(env_names=('PHYSAR_DIR',), marker='phase1_data_v6.pkl'):
    """Locate the folder that holds the pipeline files, wherever the notebook is run from.
    Order: environment variable(s) -> the current folder and its parents (and a SmartGrid_Dissertation folder next to or inside each) -> the usual places under the home folder."""
    from pathlib import Path
    cands = []
    for n in env_names:
        if os.environ.get(n): cands.append(Path(os.environ[n]).expanduser())
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents][:6]:
        cands += [p, p / 'SmartGrid_Dissertation'] + [p / n / q for n in ('smartgrid1', 'smartrid1') for q in ('', 'SmartGrid_Dissertation')]
    home = Path.home()
    for extra in ('/content', '/content/drive/MyDrive', '/content/drive/MyDrive/smartgrid1', '/content/drive/MyDrive/smartgrid1/SmartGrid_Dissertation', '/content/drive/MyDrive/smartrid1', '/content/drive/MyDrive/SmartGrid_Dissertation', '/content/drive/MyDrive/smartrid1/SmartGrid_Dissertation', '/content/drive/My Drive', '/content/drive/My Drive/smartgrid1', '/content/drive/My Drive/smartgrid1/SmartGrid_Dissertation', '/content/drive/My Drive/smartrid1/SmartGrid_Dissertation', '/kaggle/working'):
        cands += [Path(extra), Path(extra) / 'SmartGrid_Dissertation']
    for base in ('Desktop', 'Documents', 'Downloads', 'OneDrive', 'OneDrive/Desktop', 'OneDrive/Documents', ''):
        b = home / base if base else home
        cands += [b / 'SmartGrid_Dissertation'] + [b / n / q for n in ('smartgrid1', 'smartrid1') for q in ('', 'SmartGrid_Dissertation')]
    seen = set()
    for c in cands:
        try: c = c.resolve()
        except OSError: continue
        if c in seen: continue
        seen.add(c)
        if (c / marker).is_file(): return str(c)
    for root in [Path('/content'), Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')] + [home / b for b in ('Desktop', 'Documents', 'Downloads', 'OneDrive')]:   # shallow search, depth <= 3
        if not root.is_dir(): continue
        for dp, dn, fn in os.walk(root):
            if marker in fn: return dp
            if Path(dp).relative_to(root).parts.__len__() >= 3: dn[:] = []
    raise FileNotFoundError(f"Could not find {marker}. Set the environment variable PHYSAR_DIR to the folder that holds the pipeline files "
                            f"(e.g. in a cell, before running: os.environ['PHYSAR_DIR'] = r'C:\\path\\to\\SmartGrid_Dissertation'), or upload the files to the same folder as the notebook (on Colab: /content, or mount Google Drive first).")

PROJECT_DIR = find_project_dir(('NB4I_DIR', 'PHYSAR_DIR')); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = bool(int(os.environ.get('NB4I_SMOKE', '0')))
K_CAND = 12            # nearest split thresholds tried in each direction per quantity (plus the two interval ends)
MAX_ITERS = 60         # greedy rounds (one move per row per round)
CHUNK = 20000          # rows per search block
BLOCK = 400_000        # candidate rows scored per model call
N_PER_CLASS = 500      # attack rows per class evaluated (None = all attack rows)
N_BOOT = 2000
SETTINGS = {                                       # the three attacks whose effect on the derived features is studied
    'B_elec5_balance':         dict(cyber=None, eps_rel=0.05, balance=True),
    'C_cyber0p5_elec5_bal':    dict(cyber=0.5,  eps_rel=0.05, balance=True),
    'F_all93_independent_0p1': dict(cyber=0.1,  eps_rel=0.0,  balance=False, cols='all'),     # every column moved on its own, no consistency
}
T4_BUDGETS = {'0.1': 0.1, '0.5': 0.5}              # cyber-only white-box budgets for the ablation models
T4_SETTINGS = {f'T4_{k}': dict(cyber=v, eps_rel=0.0, balance=True) for k, v in T4_BUDGETS.items()}
ALL_SETTINGS = {**SETTINGS, **T4_SETTINGS}
CONS_TOL = 1e-4                                    # tolerance (in sigma units) of the derived-feature integrity check; clean rows reproduce to ~1e-8
SETTINGS_TO_RUN = list(SETTINGS)
if SMOKE_TEST: SEEDS = SEEDS[:1]; K_CAND = 4; MAX_ITERS = 6; N_PER_CLASS = 60; N_BOOT = 200

os.environ['PYTHONHASHSEED'] = '42'
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
print(f'xgboost {xgb.__version__} | search backend: {"cpu(hist)" if _cpu else "gpu(cuda,hist)"} | smoke {SMOKE_TEST} | working directory: {os.getcwd()}')

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train64 = np.asarray(data['X_train_scaled'], dtype=np.float64); X_test64 = np.asarray(data['X_test_scaled'], dtype=np.float64)
X_train = X_train64.astype(np.float32); X_test = X_test64.astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)
scaler = joblib.load(f'standard_scaler_{PIPELINE_VERSION}.pkl'); MU = np.asarray(scaler.mean_, dtype=np.float64); SC = np.asarray(scaler.scale_, dtype=np.float64)
assert len(MU) == N_FEAT

atk_te = y_test != 0; X_atk_test64 = X_test64[atk_te]; y_atk_test = y_test[atk_te]
FEAT_LO, FEAT_HI = X_train.min(0).astype(np.float64), X_train.max(0).astype(np.float64)
PB_NAMES = ['V_A', 'V_B', 'V_C', 'I_A', 'I_B', 'I_C', 'ActivePower', 'ReactivePower', 'Theta']
PB = np.array([FEATURE_COLS.index(c) for c in PB_NAMES]); DER = np.array([FEATURE_COLS.index(c) for c in ['voltage_imbalance', 'current_imbalance', 'power_factor']])
N_PHYS = len(PB)

def raw_of(Xs): return Xs[:, PB] * SC[PB] + MU[PB]
def derive(raw):
    V, I = raw[:, 0:3], raw[:, 3:6]; Vm, Im = V.mean(1), I.mean(1)
    with np.errstate(divide='ignore', invalid='ignore'):
        vimb = np.abs(V - Vm[:, None]).max(1) / np.where(Vm == 0, np.nan, Vm); cimb = np.abs(I - Im[:, None]).max(1) / np.where(Im == 0, np.nan, Im)
        S = np.hypot(raw[:, 6], raw[:, 7]); pf = raw[:, 6] / np.where(S == 0, np.nan, S)
    return np.nan_to_num(np.stack([vimb, cimb, pf], 1))
def ratio_of(raw):
    VI = (raw[:, 0:3] * raw[:, 3:6]).sum(1)
    with np.errstate(divide='ignore', invalid='ignore'): return np.hypot(raw[:, 6], raw[:, 7]) / np.where(VI == 0, np.nan, VI)
def write_back(x, raw): x[:, PB] = (raw - MU[PB]) / SC[PB]; x[:, DER] = (derive(raw) - MU[DER]) / SC[DER]

raw_train = raw_of(X_train64); RAW_LO, RAW_HI = raw_train.min(0), raw_train.max(0)
r_tr = ratio_of(raw_train); r_tr = r_tr[np.isfinite(r_tr)]; BAND_LO, BAND_HI = np.percentile(r_tr, [0.5, 99.5])
r_te = ratio_of(raw_of(X_test64)); r_te = r_te[np.isfinite(r_te)]
der_err = np.abs((derive(raw_of(X_test64)) - MU[DER]) / SC[DER] - X_test64[:, DER]).max()
print(f'Features {N_FEAT} | cyber {N_CYBER} | physical channel {N_PHYS} | attack rows in the test period {len(X_atk_test64):,}')
print(f'Power balance on the test rows: 1st {np.percentile(r_te, 1):.5f}, median {np.median(r_te):.5f}, 99th {np.percentile(r_te, 99):.5f} | genuine band [{BAND_LO:.5f}, {BAND_HI:.5f}] | max derived-feature recomputation error {der_err:.1e}')
if not SMOKE_TEST:
    assert abs(np.median(r_te) - 1.0) < 1e-3 and np.percentile(r_te, 1) > 0.998 and np.percentile(r_te, 99) < 1.002, 'power balance not reproduced -- scaler / column order problem'
    assert der_err < 1e-3, 'derived-feature recomputation does not reproduce the stored values'
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
rng0 = np.random.RandomState(0)
pick = (np.arange(len(y_atk_test)) if N_PER_CLASS is None else np.sort(np.concatenate([rng0.choice(np.flatnonzero(y_atk_test == c), min(N_PER_CLASS, int((y_atk_test == c).sum())), replace=False) for c in range(1, 6)])))
X_wb, y_wb, burst_wb = X_atk_test64[pick], y_atk_test[pick], test_burst[pick]
print(f'Evaluation rows: {len(pick):,} ({np.bincount(y_wb)[1:].tolist()} per class) in {len(np.unique(burst_wb))} of {len(np.unique(test_burst))} attack bursts')
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1) if len(nb2_tuned) > 1 else 0:.4f}')

In [ ]:
def get_cols(st): return np.arange(N_FEAT) if st.get('cols') == 'all' else CYBER_IDX
def get_thresholds(clf):
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}
class Margin:
    def __init__(self, clf): self.b = clf.get_booster()
    def __call__(self, X):
        out = []
        if len(X) == 0: return np.zeros((0, 6))
        for s in range(0, len(X), BLOCK):
            out.append(np.asarray(self.b.inplace_predict(np.ascontiguousarray(X[s:s + BLOCK], dtype=np.float32), predict_type='margin', validate_features=False)))
        return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)        # > 0  <=>  benign is the arg-max class

def make_ctx(x0):
    raw0 = raw_of(x0); rho0 = ratio_of(raw0); return raw0, np.where(np.isfinite(rho0), rho0, 1.0)
def finalize(x, raw, rho0, balance):
    """x (m, 93) float64 modified in place from the (m, 9) forged raw measurements: optional power-balance rescaling of (P, Q), then write-back with recomputed derived features."""
    if balance:
        VI = (raw[:, 0:3] * raw[:, 3:6]).sum(1); S = np.hypot(raw[:, 6], raw[:, 7])
        f = np.where(S > 0, rho0 * VI / np.where(S > 0, S, 1.0), 1.0); f = np.where(np.isfinite(f) & (f > 0), f, 1.0); raw[:, 6] *= f; raw[:, 7] *= f
    write_back(x, raw)
def phys_interval(raw0, eps):
    a, b = raw0 * (1 - eps), raw0 * (1 + eps); lo, hi = np.minimum(a, b), np.maximum(a, b)
    return np.maximum(lo, np.minimum(RAW_LO, raw0)), np.minimum(hi, np.maximum(RAW_HI, raw0))

def cand_z(xz, a, b, T, K):
    """candidate values (standardized) for one quantity: the K nearest thresholds on each side inside [a, b], plus the interval ends. Returns (row index, value)."""
    nT = len(T); idx = np.searchsorted(T, xz, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)].astype(np.float64); ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)).astype(np.float64); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xz; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xz; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V)

def one_step(margin, x, x0, raw0, rho0, g, st, TH, K):
    n = len(x); best_g = g.copy(); best_kd = np.full(n, -1, np.int8); best_j = np.zeros(n, np.int64); best_v = np.zeros(n); bufs = []; size = [0]; eps = st['eps_rel']
    lo_raw, hi_raw = phys_interval(raw0, eps)
    def flush():
        if not bufs: return
        R = np.concatenate([b[0] for b in bufs]); KD = np.concatenate([b[1] for b in bufs]); J = np.concatenate([b[2] for b in bufs]); VV = np.concatenate([b[3] for b in bufs]); bufs.clear(); size[0] = 0
        for s in range(0, len(R), BLOCK):
            r, kd, jj, v = R[s:s + BLOCK], KD[s:s + BLOCK], J[s:s + BLOCK], VV[s:s + BLOCK]; Xc = x[r].copy()
            m = np.flatnonzero(kd == 0)
            if len(m): Xc[m, jj[m]] = v[m]
            m = np.flatnonzero(kd == 1)
            if len(m):
                raw = raw_of(Xc[m]); raw[np.arange(len(m)), jj[m]] = v[m]; sub = Xc[m]; finalize(sub, raw, rho0[r[m]], st['balance']); Xc[m] = sub
            gg = gap_of(margin(Xc.astype(np.float32)))
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_kd[rr] = kd[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    def add(R, kd, j, V):
        if len(R) == 0: return
        bufs.append((R, np.full(len(R), kd, np.int8), np.full(len(R), j, np.int64), V)); size[0] += len(R)
        if size[0] >= BLOCK: flush()
    for jm in (range(N_PHYS) if eps > 0 else []):                                 # physical measurements (none when the readings are fixed)
        col = int(PB[jm]); T = TH.get(col)
        if T is None: continue
        a = (lo_raw[:, jm] - MU[col]) / SC[col]; b = (hi_raw[:, jm] - MU[col]) / SC[col]
        R, Z = cand_z(x[:, col], a, b, T, K)
        add(R, 1, jm, Z * SC[col] + MU[col])
    if st['cyber'] is not None:
        B = st['cyber']
        for col in get_cols(st):
            T = TH.get(int(col))
            if T is None: continue
            o = x0[:, col]; a = np.maximum(o - B, np.minimum(FEAT_LO[col], o)); b = np.minimum(o + B, np.maximum(FEAT_HI[col], o))
            R, Z = cand_z(x[:, col], a, b, T, K); add(R, 0, int(col), Z)
    flush(); return best_g, best_kd, best_j, best_v

def apply_moves(x, ids, kd, jj, vv, rho0, st):
    m = np.flatnonzero(kd == 0)
    if len(m): x[ids[m], jj[m]] = vv[m]
    m = np.flatnonzero(kd == 1)
    if len(m):
        sub = x[ids[m]]; raw = raw_of(sub); raw[np.arange(len(m)), jj[m]] = vv[m]; finalize(sub, raw, rho0[ids[m]], st['balance']); x[ids[m]] = sub

def whitebox(margin, X0, st, TH, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    x = X0.copy(); raw0, rho0 = make_ctx(X0); g = gap_of(margin(x.astype(np.float32))); init_evaded = g > 0; active = ~init_evaded; stalled = np.zeros(len(x), bool); log = []
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time()
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, kd, jj, vv = one_step(margin, x[ids], X0[ids], raw0[ids], rho0[ids], g[ids], st, TH, K)
            imp = kd >= 0; ii = ids[imp]; apply_moves(x, ii, kd[imp], jj[imp], vv[imp], rho0, st)
            g[ii] = gap_of(margin(x[ii].astype(np.float32))) if len(ii) else g[ii]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); log.append((it + 1, len(act), time.time() - t0))
        if verbose: print(f'      round {it + 1:>2}: active rows {len(act):>6} | {time.time() - t0:5.1f}s | evaded so far {(g > 0).mean():.3f}', flush=True)
    return x, g, init_evaded, stalled, log
print('white-box attack defined')

In [ ]:
atk_names = class_names[1:]

In [ ]:
def get_cols(st):
    c = st.get('cols')
    if isinstance(c, np.ndarray): return c
    return np.arange(N_FEAT) if c == 'all' else CYBER_IDX
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)
def fit(seed):
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X_train, y_train, verbose=False); tmp = f'_tmp_nb4i_{seed}.json'; clf.save_model(tmp); cpu = xgb.XGBClassifier(); cpu.load_model(tmp); cpu.set_params(device='cpu', n_jobs=-1); os.remove(tmp); return clf, cpu
F = dict(cyber=0.1, eps_rel=0.0, balance=False, cols='all')
DIAG_SEEDS = [42, 7, 99] if not SMOKE_TEST else SEEDS[:2]
res = {}
for seed in DIAG_SEEDS:
    t0 = time.time(); clf, cpu = fit(seed); margin = Margin(clf); TH = get_thresholds(clf)
    xa, ga, ie, stl, lg = whitebox(margin, X_wb, F, TH, verbose=False); ev = cpu.predict(xa.astype(np.float32)) == 0; ch = np.abs(xa - X_wb) > 0
    ncols_changed = ch.sum(1)
    top = np.argsort(-ch[ev].mean(0))[:8] if ev.any() else np.array([], int)
    ev_best = {}
    for name, cols in (('all93', np.arange(N_FEAT)), ('all93 minus the 3 most-used columns', np.setdiff1d(np.arange(N_FEAT), top[:3])), ('cyber60 only', CYBER_IDX)):
        s_ = dict(F, cols=cols); x2, g2, ie2, s2, l2 = whitebox(margin, X_wb, s_, TH, verbose=False); e2 = cpu.predict(x2.astype(np.float32)) == 0; ev_best[name] = (e2.mean(), np.array([e2[y_wb == c].mean() for c in range(1, 6)]), e2)
    res[seed] = dict(ev=ev, ie=ie, top=top, share=ch[ev].mean(0) if ev.any() else None, nchg=np.median(ncols_changed[ev]) if ev.any() else np.nan, variants=ev_best, f1=float(f1_score(y_test, cpu.predict(X_test), average='weighted')))
    print(f'seed {seed}: clean F1 {res[seed]["f1"]:.4f} | F evasion {ev.mean():.3f} | already benign before the attack {ie.mean():.3f} | median columns changed in an evaded row {res[seed]["nchg"]:.0f}')
    print('   columns changed most often in evaded rows:', [(FEATURE_COLS[j], round(float(res[seed]["share"][j]), 3), 'CYBER' if j in set(CYBER_IDX.tolist()) else 'ELEC') for j in top])
    for k, v in ev_best.items(): print(f'   F with {k}: evasion {v[0]:.3f} per class {np.round(v[1], 3).tolist()}')
    print(f'   {(time.time() - t0) / 60:.1f} min', flush=True); del clf, cpu, margin
print('\nEvaded-set overlap between seeds (Jaccard) and size of the union / intersection:')
ks = list(res)
for i in range(len(ks)):
    for j in range(i + 1, len(ks)):
        a, b = res[ks[i]]['ev'], res[ks[j]]['ev']; print(f'  seeds {ks[i]} vs {ks[j]}: Jaccard {(a & b).sum() / max((a | b).sum(), 1):.4f} | evaded in both {int((a & b).sum())} | in either {int((a | b).sum())} of {len(a)}')
inter = np.logical_and.reduce([res[k]['ev'] for k in ks]); never = ~np.logical_or.reduce([res[k]['ev'] for k in ks])
print(f'\nRows evaded in every seed: {inter.sum()} | rows evaded in no seed: {never.sum()} (per class: {[int(never[y_wb == c].sum()) for c in range(1, 6)]})')
print('Share of attack rows whose clean version is ALREADY benign-predicted, per seed:', {k: round(float(res[k]['ie'].mean()), 3) for k in ks})

## Reading the result
* **Same evaded set in every seed, a handful of columns used**: the evasion is carried by a few features that every model splits on (a data property, a shared weakness), not by seed-specific model structure.
* **Evasion falls when the three most-used columns are removed**: those columns carry the ceiling. Name them in the paper.
* **Evasion with the cyber-only variant is the same as all93**: the extra electrical/temporal columns are not what is used.
* **Evaded sets differ across seeds but the rate is identical**: a ceiling at the row level (for example duplicated rows), report it as such.